[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter7/vectara-agent.ipynb)

# Chapter 7: RAG Agent with Vectara

Vectara's agent API lets you build conversational RAG agents without managing retrieval infrastructure. You define a corpus with your documents, configure search tools, and the agent handles the retrieval-augmented conversation loop — including multi-turn context, tool selection, and response generation.

This notebook demonstrates the full agent lifecycle: uploading a document, creating an agent with search tools, starting a session, and querying.

**What you'll learn:**
- Upload documents to a Vectara corpus for agent-accessible retrieval
- Create an agent with corpus search tool configurations
- Manage conversational sessions with multi-turn context
- Query the agent and inspect tool-augmented responses

**Prerequisites:** Set `VECTARA_API_KEY` as an environment variable. Sign up at [vectara.com](https://vectara.com) for a free account.

## Setup

We import dependencies and configure the Vectara API key and corpus key that the agent will use for document retrieval.

In [1]:
import json
import requests
import os

Let's setup Vectara API key and corpus key

In [2]:
VECTARA_CORPUS_KEY = "hands-on-rag"
VECTARA_API_KEY = os.getenv('VECTARA_API_KEY')

## Upload Documents

Upload the GPT-2 paper using Vectara FILE_UPLOAD API endpoint

In [3]:
url = f"https://api.vectara.io/v2/corpora/{VECTARA_CORPUS_KEY}/upload_file"

payload={}
files=[
  ('file',('gpt-2-paper',open('language_models_are_unsupervised_multitask_learners.pdf','rb'),'application/octet-stream'))
]
headers = {
  'Accept': 'application/json',
  'x-api-key': VECTARA_API_KEY
}

response = requests.request("POST", url, headers=headers, data=payload, files=files)
res = response.json()
print(response.status_code)

201


## Configure and Run Agent

We define the agent's search tool configuration, create the agent with its system prompt, and start a conversational session for multi-turn interaction.

In [4]:
tool_configurations = {
    "rag_search": {
        "type": "corpora_search",
        "query_configuration": {
            "search": {
                "limit": 50,
                "corpora": [
                    {
                        "corpus_key": VECTARA_CORPUS_KEY,
                        "lexical_interpolation": 0.01,
                    }
                ],
                "context_configuration": {
                    "sentences_before": 2,
                    "sentences_after": 2
                },
                "reranker": {
                    "type": "customer_reranker",
                    "reranker_name": "Rerank_Multilingual_v1",
                    "cutoff": 0.3
                }
            },
        }
    },
}

In [5]:
agent_key = "my-agent"

create_agent_obj = {
    "key": agent_key,
    "name": "GPT-2 Agent",
    "description": "This agent handles queries and conversations related to the original GPT-2 paper about transformers.", 
    "tool_configurations": tool_configurations,
    "model": {
        "name": 'gpt-4o'
    },
    "first_step": {
        "type": "conversational",
        "instructions": [
            {
                "type": "inline",
                "name": "A Simple agent to ask questions about GPT-2 paper",
                "template": """
                    You are a chatbot that can answer questions about the GPT-2 paper.
                    Only answer questions related to GPT-2 and the original paper, and based on information provided by the tools.
                    Always respond to the user in English.
                """
            }
        ],
        "output_parser": {
            "type": "default"
        }
    }
}

In [6]:
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "x-api-key": VECTARA_API_KEY
}

# Always delete existing agent first (this also deletes all its sessions)
delete_url = f"https://api.vectara.io/v2/agents/{agent_key}"
delete_response = requests.delete(delete_url, headers=headers)
if delete_response.status_code == 204:
    print(f"Deleted existing agent '{agent_key}'")
elif delete_response.status_code == 404:
    print(f"Agent '{agent_key}' does not exist, creating new one")
else:
    print(f"Delete returned: {delete_response.status_code}")

# Create the agent
url = "https://api.vectara.io/v2/agents"
response = requests.post(url, json=create_agent_obj, headers=headers)
print(f"Create agent status: {response.status_code}")
if response.status_code != 201:
    print(response.text)

Deleted existing agent 'my-agent'
Create agent status: 201


In [7]:
import time

# Use unique session key and name to avoid conflicts with orphaned sessions
timestamp = int(time.time())
session_key = f"session-{timestamp}"
session_name = f"GPT-2 session {timestamp}"
print(f"Using session: key={session_key}, name={session_name}")

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': VECTARA_API_KEY,
}

# Create the session
url = f"https://api.vectara.io/v2/agents/{agent_key}/sessions"
payload = {
    "key": session_key,
    "name": session_name,
    "description": "Help users with questions about GPT-2",
    "enabled": True
}

response = requests.post(url, headers=headers, json=payload)
print(f"Create session status: {response.status_code}")
if response.status_code != 201:
    print(response.text)

Using session: key=session-1770752791, name=GPT-2 session 1770752791
Create session status: 201


## Query the Agent

With the agent and session ready, we send a question and inspect the tool-augmented response to verify the agent retrieves and synthesizes information from the uploaded document.

In [8]:
url = f"https://api.vectara.io/v2/agents/{agent_key}/sessions/{session_key}/events"
query = "What is GPT-2?"

payload = {
    "type": "input_message",
    "messages": [
        {
            "type": "text",
            "content": query,
        }
    ],
    "stream_response": False
}
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': VECTARA_API_KEY,
}

response = requests.post(url, headers=headers, json=payload)
print(f"Events status: {response.status_code}")

res = response.json()
if 'events' in res:
    print(res['events'][-1])
else:
    print(f"Error: {res}")

Events status: 201
{'id': 'aev_bb9a0e5e-be23-450e-bd1b-382dabe306e0', 'session_key': 'session-1770752791', 'type': 'agent_output', 'content': "GPT-2 is a large-scale, unsupervised language model developed by OpenAI, which consists of 1.5 billion parameters. It achieves state-of-the-art results on 7 out of 8 tested language modeling datasets in a zero-shot setting. Despite its substantial capacity, GPT-2 still exhibits underfitting on the WebText dataset it was trained on. The model's performance is notable for not requiring supervised training, although it sometimes uses simple retrieval heuristics. It shows promise in performing zero-shot task transfers, and its capacity significantly improves its performance across various tasks. Samples generated by GPT-2 demonstrate coherent paragraphs of text, reflecting these improvements.\n\nIn summary, GPT-2 is designed to perform multiple tasks without explicit task-specific training data, leveraging a large and diverse dataset for training.",

In [9]:
if 'events' in res:
    print(res['events'][-1]['content'])

GPT-2 is a large-scale, unsupervised language model developed by OpenAI, which consists of 1.5 billion parameters. It achieves state-of-the-art results on 7 out of 8 tested language modeling datasets in a zero-shot setting. Despite its substantial capacity, GPT-2 still exhibits underfitting on the WebText dataset it was trained on. The model's performance is notable for not requiring supervised training, although it sometimes uses simple retrieval heuristics. It shows promise in performing zero-shot task transfers, and its capacity significantly improves its performance across various tasks. Samples generated by GPT-2 demonstrate coherent paragraphs of text, reflecting these improvements.

In summary, GPT-2 is designed to perform multiple tasks without explicit task-specific training data, leveraging a large and diverse dataset for training.
